In [1]:
# Run this cell first
import os
import sys

project_root = os.getcwd()
if os.path.basename(project_root) == "notebooks":
    project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

Project root: c:\Users\Administrator\Desktop\ai_doc_classifier


In [2]:
import os

manifest_path = os.path.join(project_root, "data", "training_manifest.json")
print("Looking for manifest at:", manifest_path)
print("Exists?", os.path.exists(manifest_path))

if os.path.exists(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        print("First 300 chars of manifest:\n", f.read()[:300])
else:
    print("Manifest file not found. Check folder structure.")


Looking for manifest at: c:\Users\Administrator\Desktop\ai_doc_classifier\data\training_manifest.json
Exists? True
First 300 chars of manifest:
 {
  "education_resume.pdf": "Education",
  "finance_resume.pdf": "Finance",
  "health_resume.pdf": "Healthcare",
  "tech_resume.pdf": "Tech",
  "accounts_resume.pdf": "Accounting",
  "analyst_resume.pdf": "Analyst",
  "engineering_resume.pdf": "Engineering",
  "marketing_resume.pdf": "Marketing",
  


In [3]:
# Imports
from docx import text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib
import os, json
from pathlib import Path

from utils.extract import extract_text_pdf
from utils.preprocess import clean_text


def validate_resume_text(text: str) -> bool:
    if not text.strip():
        return False
    if sum(c.isalpha() for c in text) < 20:  # too few letters
        return False
    return True


def load_resume_texts(data_dir, manifest_path):
    texts, labels = [], []
    manifest = json.loads(Path(manifest_path).read_text(encoding="utf-8"))

    for filename, label in manifest.items():
        resume_path = os.path.join(data_dir, filename)
        if not os.path.exists(resume_path):
            continue

        text = extract_text_pdf(resume_path)

        # 🔍 Debug: print first 200 characters to check extraction
        # print(f"\n--- {filename} ---")
        # print(text[:200])

        cleaned = clean_text(text)

        # NEW: handle gibberish/empty
        if not validate_resume_text(cleaned):
            label = "Other"

        texts.append(cleaned)
        labels.append(label)

    return texts, labels


def train_resume_classifier(texts, labels):
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(texts)

    model = LogisticRegression(max_iter=1000)
    model.fit(X, labels)

    return model, vectorizer


def save_artifacts(model, vectorizer, models_dir):
    os.makedirs(models_dir, exist_ok=True)
    joblib.dump(model, os.path.join(models_dir, "resume_classifier.pkl"))
    joblib.dump(vectorizer, os.path.join(models_dir, "vectorizer.pkl"))


# Step 1: Load resumes using manifest
data_dir = os.path.join(project_root, "data", "raw")
manifest_path = os.path.join(project_root, "data", "training_manifest.json")  # ✅ corrected
print("Manifest path:", manifest_path)
print("Exists?", os.path.exists(manifest_path))

texts, labels = load_resume_texts(data_dir, manifest_path)

# Step 2: Train model
model, vectorizer = train_resume_classifier(texts, labels)

# Step 3: Save both model and vectorizer
models_dir = os.path.join(project_root, "models")
save_artifacts(model, vectorizer, models_dir)

print("Training complete. Model and vectorizer saved.")

# Step 4: Test predictions inside notebook
test_samples = [
    "Experienced Python developer with Django and Flask",
    "Financial analyst skilled in Excel and accounting",
    "Registered nurse with hospital experience",
    "Teacher with classroom management and curriculum design",
]

for sample in test_samples:
    pred = model.predict(vectorizer.transform([sample]))
    print(f"Text: {sample}\nPredicted Category: {pred[0]}\n")

# Step 5: Load saved artifacts for later reuse
models_dir = os.path.join(project_root, "models")
model = joblib.load(os.path.join(models_dir, "resume_classifier.pkl"))
vectorizer = joblib.load(os.path.join(models_dir, "vectorizer.pkl"))

print("Loaded saved model and vectorizer.")


Manifest path: c:\Users\Administrator\Desktop\ai_doc_classifier\data\training_manifest.json
Exists? True
Training complete. Model and vectorizer saved.
Text: Experienced Python developer with Django and Flask
Predicted Category: Tech

Text: Financial analyst skilled in Excel and accounting
Predicted Category: Finance

Text: Registered nurse with hospital experience
Predicted Category: Healthcare

Text: Teacher with classroom management and curriculum design
Predicted Category: Education

Loaded saved model and vectorizer.


In [4]:
# Reuse saved model and vectorizer for custom predictions
import joblib
import os


def load_saved_artifacts(models_dir=None):
    if models_dir is None:
        models_dir = os.path.join(project_root, "models")

    model = joblib.load(os.path.join(models_dir, "resume_classifier.pkl"))
    vectorizer = joblib.load(os.path.join(models_dir, "vectorizer.pkl"))
    return model, vectorizer


def predict_category(text, model=None, vectorizer=None):
    if model is None or vectorizer is None:
        model, vectorizer = load_saved_artifacts()

    # NEW: validate before predicting
    if not validate_resume_text(text):
        return "Other"

    return model.predict(vectorizer.transform([text]))[0]


custom_samples = [
    "Certified AWS engineer with cloud migration experience",
    "Budget analyst with forecasting and variance reporting",
    "Patient care coordinator in a community clinic",
    "Curriculum specialist focused on instructional design",
]

model, vectorizer = load_saved_artifacts()
for sample in custom_samples:
    print(f"Text: {sample}\nPredicted Category: {predict_category(sample, model, vectorizer)}\n")

Text: Certified AWS engineer with cloud migration experience
Predicted Category: Tech

Text: Budget analyst with forecasting and variance reporting
Predicted Category: Analyst

Text: Patient care coordinator in a community clinic
Predicted Category: Healthcare

Text: Curriculum specialist focused on instructional design
Predicted Category: Education



In [5]:
# Quick one-off test: extract text from a single resume
resume_path = os.path.join(project_root, "data", "raw", "tech_resume.pdf")  # change filename as needed
text = extract_text_pdf(resume_path)

print("\n--- Extracted text preview ---")
print(text[:300])  # show first 300 characters

cleaned = clean_text(text)
print("\n--- Cleaned text preview ---")
print(cleaned[:300])



--- Extracted text preview ---
John Doe 
Software Engineer | Python Developer 
Email: johndoe@example.com | Phone: +1 555-123-4567 
Location: Nairobi, Kenya 
 
Summary 
Highly motivated software engineer with 5+ years of experience in full-stack development, 
specializing in Python, Django, and cloud deployment. Passionate about 

--- Cleaned text preview ---
john doe software engineer python developer email johndoe example com phone 1 555 123 4567 location nairobi kenya summary highly motivated software engineer with 5 years of experience in full stack development specializing in python django and cloud deployment passionate about building scalable appl
